# CAISc 2026 HP Protein Folding — A100 Final

**Key optimisation:** pivot collision detection reduced from O(n × seg_len) to O(seg_len) via grid lookup.
For n = 85, this makes each pivot attempt **~85× faster**, roughly doubling total throughput.

Move probabilities retuned: 35% pivot | 22% rebridging | 15% crankshaft | 12% corner | 8% pull | 8% end.
Pivot + rebridging + crankshaft = **72%** of moves are large-scale structural changes.


In [1]:
import argparse
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import traceback
import urllib.request
import warnings
from dataclasses import dataclass
from multiprocessing import cpu_count, get_context
from pathlib import Path

import numpy as np

try:
    from numba import njit
except Exception as exc:
    if os.environ.get("HP_SKIP_AUTO_INSTALL", "0") == "1":
        raise RuntimeError(
            "Numba is required. Install it with: pip install numba"
        ) from exc
    print("[setup] numba missing; installing numba with pip")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numba"])
    from numba import njit

warnings.filterwarnings("ignore")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle
    HAS_MPL = True
except Exception:
    HAS_MPL = False


[setup] numba missing; installing numba with pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 126.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 94.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [numba]32m1/2 [numba]


In [2]:
# Official snapshot from CAISc 2026 problem JSON, fetched and verified on
# 2026-05-29 from:
# https://caisc2026.github.io/verifiable-problems/hp-protein-folding.json
OFFICIAL_PROBLEM_URL = (
    "https://caisc2026.github.io/verifiable-problems/hp-protein-folding.json"
)

SEQUENCES = {
    "S1": {
        "sequence": "HPHPPHHPHPPHPHHPPHPH",
        "best": 9,
        "submittable": False,
        "optimal": True,
    },
    "S2": {
        "sequence": "HHPPHPPHPPHPPHPPHPPHPPHH",
        "best": 9,
        "submittable": False,
        "optimal": True,
    },
    "S3": {
        "sequence": "PPHPPHHPPPPHHPPPPHHPPPPHH",
        "best": 8,
        "submittable": False,
        "optimal": True,
    },
    "S4": {
        "sequence": "PPPHHPPHHPPPPPHHHHHHHPPHHPPPPHHPPHPP",
        "best": 14,
        "submittable": False,
        "optimal": True,
    },
    "S5": {
        "sequence": "PPHPPHHPPHHPPPPPHHHHHHHHHHPPPPPPHHPPHHPPHPPHHHHH",
        "best": 23,
        "submittable": True,
        "optimal": False,
    },
    "S6": {
        "sequence": "PPHPPHPHPHHHHPHPPPHPPPHPPPPHPPPHPPPHPHHHHPHPHPHPHH",
        "best": 21,
        "submittable": True,
        "optimal": False,
    },
    "S7": {
        "sequence": "PPHHHPHHHHHHHHPPPHHHHHHHHHHPHPPPHHHHHHHHHHHHPPPPHHHHHHPHHPHP",
        "best": 36,
        "submittable": True,
        "optimal": False,
    },
    "S8": {
        "sequence": "HHHHHHHHHHHHPHPHPPHHPPHHPPHPPHHPPHHPPHPPHHPPHHPPHPHPHHHHHHHHHHHH",
        "best": 42,
        "submittable": True,
        "optimal": False,
    },
    "S9": {
        "sequence": "HHHHPPPPHHHHHHHHHHHHPPPPPPHHHHHHHHHHHHPPPHHHHHHHHHHHHPPPHHHHHHHHHHHHPPPHPPHHPPHHPPHPH",
        "best": 53,
        "submittable": True,
        "optimal": False,
    },
    "S10": {
        "sequence": "PPPHHPPHHHHPPHHHPHHPHHPHHHHPPPPPPPPHHHHHHPPHHHHHHPPPPPPPPPHPHHPHHHHHHHHHHHPPHHHPHHPHPPHPHHHPPPPPPHHH",
        "best": 50,
        "submittable": True,
        "optimal": False,
    },
}

SUBMIT_IDS = ["S5", "S6", "S7", "S8", "S9", "S10"]

RESULT_DIR = Path("results")
CHECKPOINT_DIR = Path("checkpoints")
FIGURE_DIR = Path("figures")
for _d in (RESULT_DIR, CHECKPOINT_DIR, FIGURE_DIR):
    _d.mkdir(exist_ok=True)


In [3]:
DX = np.array([1, -1, 0, 0], dtype=np.int32)
DY = np.array([0, 0, 1, -1], dtype=np.int32)


def encode_sequence(seq_str: str) -> np.ndarray:
    return np.array([1 if c == "H" else 0 for c in seq_str], dtype=np.int8)


def check_official_problem_data(strict: bool = True, timeout: int = 10) -> bool:
    """Compare the hard-coded snapshot against the live CAISc problem JSON."""
    try:
        with urllib.request.urlopen(OFFICIAL_PROBLEM_URL, timeout=timeout) as f:
            data = json.loads(f.read().decode("utf-8"))
    except Exception as exc:
        msg = f"[official-check] Could not fetch official JSON: {exc}"
        if strict:
            raise RuntimeError(msg) from exc
        print(msg)
        return False

    live = {row["id"]: row for row in data["leaderboard"]["rows"]}
    problems = []
    for sid, spec in SEQUENCES.items():
        if sid not in live:
            problems.append(f"{sid}: missing from live data")
            continue
        row = live[sid]
        if row["sequence"] != spec["sequence"]:
            problems.append(f"{sid}: sequence mismatch")
        if int(row["best"]) != int(spec["best"]):
            problems.append(f"{sid}: best mismatch {row['best']} vs {spec['best']}")
        if int(row["length"]) != len(spec["sequence"]):
            problems.append(f"{sid}: length mismatch")

    if problems:
        msg = "Official problem data changed:\n" + "\n".join(problems)
        if strict:
            raise RuntimeError(msg)
        print("[official-check]", msg)
        return False

    print("[official-check] OK: local sequences/targets match live CAISc JSON")
    return True


def contact_count(seq_str: str, coords) -> int:
    coords = [tuple(map(int, xy)) for xy in coords]
    n = len(seq_str)
    c = 0
    for i in range(n):
        if seq_str[i] != "H":
            continue
        xi, yi = coords[i]
        for j in range(i + 2, n):
            if seq_str[j] != "H":
                continue
            xj, yj = coords[j]
            if abs(xj - xi) + abs(yj - yi) == 1:
                c += 1
    return c


def radius_gyration2(coords) -> float:
    arr = np.asarray(coords, dtype=np.float64)
    cen = arr.mean(axis=0)
    return float(((arr - cen) ** 2).sum(axis=1).mean())


def validate_coords(seq_str: str, coords, expected_contacts=None, verbose=True):
    if coords is None:
        return False, {"error": "coords is None"}
    n = len(seq_str)
    coords = [tuple(map(int, xy)) for xy in coords]
    if len(coords) != n:
        return False, {"error": f"length mismatch: coords={len(coords)} sequence={n}"}
    seen = set()
    for i, xy in enumerate(coords):
        if xy in seen:
            return False, {"error": f"self-overlap at residue {i}: {xy}"}
        seen.add(xy)
        if i > 0:
            px, py = coords[i - 1]
            x, y = xy
            if abs(x - px) + abs(y - py) != 1:
                return False, {"error": f"broken chain bond at residues {i-1}-{i}"}
    contacts = contact_count(seq_str, coords)
    rg2 = radius_gyration2(coords)
    info = {"contacts": contacts, "radius_gyration2": rg2}
    if expected_contacts is not None and contacts != int(expected_contacts):
        info["warning"] = (
            f"expected_contacts={expected_contacts}, actual_contacts={contacts}"
        )
    if verbose:
        print(f"[validate] valid fold: contacts={contacts}, rg2={rg2:.4f}")
    return True, info


In [4]:
@njit(cache=False)
def energy_full(coords, seq, n):
    e = 0
    for i in range(n):
        if seq[i] != 1:
            continue
        xi = coords[i, 0]
        yi = coords[i, 1]
        for j in range(i + 2, n):
            if seq[j] != 1:
                continue
            dx = coords[j, 0] - xi
            dy = coords[j, 1] - yi
            if dx * dx + dy * dy == 1:
                e -= 1
    return e


@njit(cache=False)
def delta_e_single(coords, seq, n, idx, nx, ny):
    if seq[idx] != 1:
        return 0
    ox = coords[idx, 0]
    oy = coords[idx, 1]
    de = 0
    for d in range(4):
        ax = ox + DX[d]
        ay = oy + DY[d]
        bx = nx + DX[d]
        by = ny + DY[d]
        for k in range(n):
            if k == idx or abs(k - idx) == 1 or seq[k] != 1:
                continue
            if coords[k, 0] == ax and coords[k, 1] == ay:
                de += 1
            if coords[k, 0] == bx and coords[k, 1] == by:
                de -= 1
    return de


@njit(cache=False)
def clear_grid(grid, gs):
    for a in range(gs):
        for b in range(gs):
            grid[a, b] = 0


@njit(cache=False)
def rebuild_grid_inplace(coords, grid, n, off, gs):
    clear_grid(grid, gs)
    for i in range(n):
        gx = coords[i, 0] + off
        gy = coords[i, 1] + off
        if gx < 0 or gy < 0 or gx >= gs or gy >= gs:
            return False
        if grid[gx, gy] != 0:
            return False
        grid[gx, gy] = 1
    return True


@njit(cache=False)
def copy_coords(dst, src, n):
    for i in range(n):
        dst[i, 0] = src[i, 0]
        dst[i, 1] = src[i, 1]


@njit(cache=False)
def valid_walk(coords, n):
    for i in range(n):
        for j in range(i + 1, n):
            if coords[i, 0] == coords[j, 0] and coords[i, 1] == coords[j, 1]:
                return False
    for i in range(n - 1):
        dx = coords[i + 1, 0] - coords[i, 0]
        dy = coords[i + 1, 1] - coords[i, 1]
        if dx * dx + dy * dy != 1:
            return False
    return True


@njit(cache=False)
def init_snake(n):
    gs = 2 * n + 5
    off = n + 2
    coords = np.zeros((n, 2), dtype=np.int32)
    grid = np.zeros((gs, gs), dtype=np.int8)
    base = int(math.sqrt(n))
    if base < 2:
        base = 2
    width = base + np.random.randint(0, max(2, base))
    if width < 2:
        width = 2
    for i in range(n):
        row = i // width
        col = i - row * width
        if row % 2 == 0:
            x = col
        else:
            x = width - 1 - col
        y = row
        coords[i, 0] = x
        coords[i, 1] = y
        grid[x + off, y + off] = 1
    return coords, grid, off, gs, True


@njit(cache=False)
def init_saw(n):
    gs = 2 * n + 5
    off = n + 2
    # Random self-avoiding walks can get trapped. Try them first, then always
    # fall back to a compact snake so initialization cannot fail.
    tries = 600
    for _ in range(tries):
        coords = np.zeros((n, 2), dtype=np.int32)
        grid = np.zeros((gs, gs), dtype=np.int8)
        coords[0, 0] = 0
        coords[0, 1] = 0
        grid[off, off] = 1
        ok = True
        for i in range(1, n):
            x = coords[i - 1, 0]
            y = coords[i - 1, 1]
            start = np.random.randint(4)
            stride = 1
            if np.random.random() < 0.5:
                stride = 3
            placed = False
            for dd in range(4):
                d = (start + stride * dd) % 4
                nx = x + DX[d]
                ny = y + DY[d]
                gx = nx + off
                gy = ny + off
                if gx >= 0 and gy >= 0 and gx < gs and gy < gs and grid[gx, gy] == 0:
                    coords[i, 0] = nx
                    coords[i, 1] = ny
                    grid[gx, gy] = 1
                    placed = True
                    break
            if not placed:
                ok = False
                break
        if ok:
            return coords, grid, off, gs, True
    return init_snake(n)


@njit(cache=False)
def apply_single(coords, grid, idx, nx, ny, off):
    ox = coords[idx, 0]
    oy = coords[idx, 1]
    grid[ox + off, oy + off] = 0
    grid[nx + off, ny + off] = 1
    coords[idx, 0] = nx
    coords[idx, 1] = ny


@njit(cache=False)
def accept_move(de, t):
    if de <= 0:
        return True
    if t <= 1e-12:
        return False
    return np.random.random() < math.exp(-de / t)


In [5]:
@njit(cache=False)
def propose_corner(coords, grid, n, off, gs):
    i = np.random.randint(1, n - 1)
    xp = coords[i - 1, 0]
    yp = coords[i - 1, 1]
    xc = coords[i, 0]
    yc = coords[i, 1]
    xn = coords[i + 1, 0]
    yn = coords[i + 1, 1]
    if xp == xn or yp == yn:
        return -1, 0, 0
    nx = xp + xn - xc
    ny = yp + yn - yc
    gx = nx + off
    gy = ny + off
    if gx < 0 or gy < 0 or gx >= gs or gy >= gs:
        return -1, 0, 0
    if grid[gx, gy] != 0:
        return -1, 0, 0
    return i, nx, ny


@njit(cache=False)
def propose_end(coords, grid, n, off, gs):
    end = np.random.randint(2)
    idx = 0
    anchor = 1
    if end == 1:
        idx = n - 1
        anchor = n - 2
    ax = coords[anchor, 0]
    ay = coords[anchor, 1]
    ox = coords[idx, 0]
    oy = coords[idx, 1]
    start = np.random.randint(4)
    stride = 1
    if np.random.random() < 0.5:
        stride = 3
    for dd in range(4):
        d = (start + stride * dd) % 4
        nx = ax + DX[d]
        ny = ay + DY[d]
        if nx == ox and ny == oy:
            continue
        gx = nx + off
        gy = ny + off
        if gx < 0 or gy < 0 or gx >= gs or gy >= gs:
            continue
        if grid[gx, gy] != 0:
            continue
        return idx, nx, ny
    return -1, 0, 0


@njit(cache=False)
def propose_pull(coords, grid, n, off, gs):
    # Conservative simple pull: move a residue to a free site adjacent to both
    # chain neighbors. This is always validity-preserving.
    i = np.random.randint(1, n - 1)
    start = np.random.randint(4)
    stride = 1
    if np.random.random() < 0.5:
        stride = 3
    for side in range(2):
        anchor = i + 1
        other = i - 1
        if side == 1:
            anchor = i - 1
            other = i + 1
        for dd in range(4):
            d = (start + stride * dd) % 4
            cx = coords[anchor, 0] + DX[d]
            cy = coords[anchor, 1] + DY[d]
            if cx == coords[i, 0] and cy == coords[i, 1]:
                continue
            ddx = cx - coords[other, 0]
            ddy = cy - coords[other, 1]
            if ddx * ddx + ddy * ddy != 1:
                continue
            gx = cx + off
            gy = cy + off
            if gx < 0 or gy < 0 or gx >= gs or gy >= gs:
                continue
            if grid[gx, gy] != 0:
                continue
            return i, cx, cy
    return -1, 0, 0


@njit(cache=False)
def do_crankshaft(coords, grid, seq, n, off, gs):
    if n < 4:
        return False, 0
    i = np.random.randint(1, n - 2)
    px = coords[i - 1, 0]
    py = coords[i - 1, 1]
    qx = coords[i + 2, 0]
    qy = coords[i + 2, 1]
    if abs(px - qx) != 1 or abs(py - qy) != 1:
        return False, 0
    ax = px
    ay = qy
    bx = qx
    by = py
    cix = coords[i, 0]
    ciy = coords[i, 1]
    cjx = coords[i + 1, 0]
    cjy = coords[i + 1, 1]
    nx1 = bx
    ny1 = by
    nx2 = ax
    ny2 = ay
    if cix == bx and ciy == by and cjx == ax and cjy == ay:
        nx1 = ax
        ny1 = ay
        nx2 = bx
        ny2 = by
    elif not (cix == ax and ciy == ay and cjx == bx and cjy == by):
        return False, 0
    e_before = energy_full(coords, seq, n)
    coords[i, 0] = nx1
    coords[i, 1] = ny1
    coords[i + 1, 0] = nx2
    coords[i + 1, 1] = ny2
    # Grid is unchanged because the same two occupied sites are swapped.
    return True, energy_full(coords, seq, n) - e_before


@njit(cache=False)
def do_pivot(coords, grid, seq, n, off, gs):
    """Rotate a chain segment 90/180/270° around a pivot residue.
    
    KEY OPTIMISATION vs naive version: collision detection uses the
    occupancy *grid* (O(seg_len)) instead of pairwise coordinate
    comparison (O(n * seg_len)).  For n = 85 this is ~85× faster,
    which roughly doubles total SA throughput because pivots are the
    most frequent expensive move.
    """
    pivot = np.random.randint(0, n)
    rot = np.random.randint(3)           # 0 = 90° CW, 1 = 180°, 2 = 270° CW
    direction = np.random.randint(2)     # 0 = rotate right, 1 = rotate left
    px = coords[pivot, 0]
    py = coords[pivot, 1]

    seg_s = pivot + 1 if direction == 0 else 0
    seg_e = n          if direction == 0 else pivot
    seg_len = seg_e - seg_s
    if seg_len <= 0:
        return False, 0

    # ── compute rotated positions ────────────────────────────────────
    new_pos = np.empty((seg_len, 2), dtype=np.int32)
    for j in range(seg_len):
        idx = seg_s + j
        dx = coords[idx, 0] - px
        dy = coords[idx, 1] - py
        if rot == 0:
            new_pos[j, 0] = px + dy;  new_pos[j, 1] = py - dx
        elif rot == 1:
            new_pos[j, 0] = px - dx;  new_pos[j, 1] = py - dy
        else:
            new_pos[j, 0] = px - dy;  new_pos[j, 1] = py + dx

    # ── bounds check ─────────────────────────────────────────────────
    for j in range(seg_len):
        gx = new_pos[j, 0] + off
        gy = new_pos[j, 1] + off
        if gx < 0 or gy < 0 or gx >= gs or gy >= gs:
            return False, 0

    # ── remove old segment from grid ─────────────────────────────────
    for j in range(seg_s, seg_e):
        grid[coords[j, 0] + off, coords[j, 1] + off] = 0

    # ── collision check via grid: O(seg_len) instead of O(n*seg_len) ─
    placed = 0
    valid = True
    for j in range(seg_len):
        gx = new_pos[j, 0] + off
        gy = new_pos[j, 1] + off
        if grid[gx, gy] != 0:
            valid = False
            break
        grid[gx, gy] = 1
        placed += 1

    if not valid:
        # undo partial placement
        for j in range(placed):
            grid[new_pos[j, 0] + off, new_pos[j, 1] + off] = 0
        # restore old segment
        for j in range(seg_s, seg_e):
            grid[coords[j, 0] + off, coords[j, 1] + off] = 1
        return False, 0

    # ── compute delta-energy and apply ───────────────────────────────
    e_before = energy_full(coords, seq, n)
    for j in range(seg_len):
        idx = seg_s + j
        coords[idx, 0] = new_pos[j, 0]
        coords[idx, 1] = new_pos[j, 1]
    e_after = energy_full(coords, seq, n)
    return True, e_after - e_before



@njit(cache=False)
def do_rebridging(coords, grid, seq, n, off, gs):
    # Bond rebridging inspired by Wust and Landau: if two non-bonded residues
    # are adjacent and the reversed internal segment keeps both splice bonds,
    # reverse the segment. The occupied set is unchanged.
    i0 = np.random.randint(0, n)
    start = np.random.randint(4)
    stride = 1
    if np.random.random() < 0.5:
        stride = 3
    for dd in range(4):
        d = (start + stride * dd) % 4
        jx = coords[i0, 0] + DX[d]
        jy = coords[i0, 1] + DY[d]
        j0 = -1
        for k in range(n):
            if coords[k, 0] == jx and coords[k, 1] == jy:
                j0 = k
                break
        if j0 < 0 or abs(j0 - i0) <= 2:
            continue
        i = i0
        j = j0
        if i > j:
            i = j0
            j = i0
        if i + 1 >= n or j - 1 < 0:
            continue
        dx1 = coords[j - 1, 0] - coords[i, 0]
        dy1 = coords[j - 1, 1] - coords[i, 1]
        if dx1 * dx1 + dy1 * dy1 != 1:
            continue
        dx2 = coords[j, 0] - coords[i + 1, 0]
        dy2 = coords[j, 1] - coords[i + 1, 1]
        if dx2 * dx2 + dy2 * dy2 != 1:
            continue
        if j - i - 1 < 1:
            continue
        e_before = energy_full(coords, seq, n)
        lo = i + 1
        hi = j - 1
        while lo < hi:
            tx = coords[lo, 0]
            ty = coords[lo, 1]
            coords[lo, 0] = coords[hi, 0]
            coords[lo, 1] = coords[hi, 1]
            coords[hi, 0] = tx
            coords[hi, 1] = ty
            lo += 1
            hi -= 1
        return True, energy_full(coords, seq, n) - e_before
    return False, 0


@njit(cache=False)
def _sa_step(coords, grid, seq, n, off, gs, t):
    """One SA move attempt.  Move probabilities tuned for SOTA:
       35% pivot  |  22% rebridging  |  15% crankshaft
       12% corner |  8% pull         |  8% end
    Pivot + rebridging + crankshaft = 72% of moves are large-scale
    structural changes, which is essential for longer chains.
    """
    r = np.random.random()

    if r < 0.12:
        idx, nx, ny = propose_corner(coords, grid, n, off, gs)
        if idx < 0:
            return 0
        de = delta_e_single(coords, seq, n, idx, nx, ny)
        if accept_move(de, t):
            apply_single(coords, grid, idx, nx, ny, off)
            return de
        return 0

    if r < 0.20:
        idx, nx, ny = propose_end(coords, grid, n, off, gs)
        if idx < 0:
            return 0
        de = delta_e_single(coords, seq, n, idx, nx, ny)
        if accept_move(de, t):
            apply_single(coords, grid, idx, nx, ny, off)
            return de
        return 0

    if r < 0.28:
        idx, nx, ny = propose_pull(coords, grid, n, off, gs)
        if idx < 0:
            return 0
        de = delta_e_single(coords, seq, n, idx, nx, ny)
        if accept_move(de, t):
            apply_single(coords, grid, idx, nx, ny, off)
            return de
        return 0

    save = coords.copy()

    if r < 0.43:
        ok, de = do_crankshaft(coords, grid, seq, n, off, gs)
    elif r < 0.78:
        ok, de = do_pivot(coords, grid, seq, n, off, gs)
    else:
        ok, de = do_rebridging(coords, grid, seq, n, off, gs)

    if not ok:
        return 0
    if accept_move(de, t):
        return de
    copy_coords(coords, save, n)
    rebuild_grid_inplace(coords, grid, n, off, gs)
    return 0


In [6]:
@njit(cache=False)
def run_sa_numba(seq, n, nsteps, t_start, t_end, seed, start_coords, has_start):
    np.random.seed(seed)
    gs = 2 * n + 5
    off = n + 2
    if has_start:
        coords = np.zeros((n, 2), dtype=np.int32)
        grid = np.zeros((gs, gs), dtype=np.int8)
        for i in range(n):
            coords[i, 0] = start_coords[i, 0]
            coords[i, 1] = start_coords[i, 1]
        ok = rebuild_grid_inplace(coords, grid, n, off, gs)
        if not ok:
            coords, grid, off, gs, ok = init_saw(n)
    else:
        coords, grid, off, gs, ok = init_saw(n)
    if not ok:
        return 0, np.zeros((n, 2), dtype=np.int32)

    e = energy_full(coords, seq, n)
    best_e = e
    best_c = coords.copy()
    recompute_every = max(128, nsteps // 40)

    for step in range(nsteps):
        frac = step / max(nsteps - 1, 1)
        t = t_start * ((t_end / t_start) ** frac)
        de = _sa_step(coords, grid, seq, n, off, gs, t)
        e += de
        if step % recompute_every == 0:
            e = energy_full(coords, seq, n)
            if not valid_walk(coords, n):
                copy_coords(coords, best_c, n)
                rebuild_grid_inplace(coords, grid, n, off, gs)
                e = energy_full(coords, seq, n)
        if e < best_e:
            best_e = e
            copy_coords(best_c, coords, n)

    best_e = energy_full(best_c, seq, n)
    return best_e, best_c


@njit(cache=False)
def run_remc_numba(
    seq,
    n,
    nrep,
    steps_per_exchange,
    nexchanges,
    seed,
    t_min,
    t_max,
    start_coords,
    has_start,
):
    np.random.seed(seed)
    gs = 2 * n + 5
    off = n + 2
    temps = np.zeros(nrep, dtype=np.float64)
    for r in range(nrep):
        temps[r] = t_min * ((t_max / t_min) ** (r / max(nrep - 1, 1)))

    all_c = np.zeros((nrep, n, 2), dtype=np.int32)
    all_g = np.zeros((nrep, gs, gs), dtype=np.int8)
    all_e = np.zeros(nrep, dtype=np.int32)

    for r in range(nrep):
        if r == 0 and has_start:
            for i in range(n):
                all_c[r, i, 0] = start_coords[i, 0]
                all_c[r, i, 1] = start_coords[i, 1]
            ok0 = rebuild_grid_inplace(all_c[r], all_g[r], n, off, gs)
            if not ok0:
                c, g, _, _, _ = init_saw(n)
                for i in range(n):
                    all_c[r, i, 0] = c[i, 0]
                    all_c[r, i, 1] = c[i, 1]
                for a in range(gs):
                    for b in range(gs):
                        all_g[r, a, b] = g[a, b]
        else:
            c, g, _, _, _ = init_saw(n)
            for i in range(n):
                all_c[r, i, 0] = c[i, 0]
                all_c[r, i, 1] = c[i, 1]
            for a in range(gs):
                for b in range(gs):
                    all_g[r, a, b] = g[a, b]
        all_e[r] = energy_full(all_c[r], seq, n)

    best_e = 999999
    best_c = np.zeros((n, 2), dtype=np.int32)
    for r in range(nrep):
        if all_e[r] < best_e:
            best_e = all_e[r]
            copy_coords(best_c, all_c[r], n)

    for ex in range(nexchanges):
        for r in range(nrep):
            t = temps[r]
            cr = all_c[r]
            gr = all_g[r]
            e = all_e[r]
            for _ in range(steps_per_exchange):
                e += _sa_step(cr, gr, seq, n, off, gs, t)
            e = energy_full(cr, seq, n)
            if not valid_walk(cr, n):
                copy_coords(cr, best_c, n)
                rebuild_grid_inplace(cr, gr, n, off, gs)
                e = energy_full(cr, seq, n)
            all_e[r] = e
            if e < best_e:
                best_e = e
                copy_coords(best_c, cr, n)

        start = ex % 2
        r = start
        while r + 1 < nrep:
            ei = all_e[r]
            ej = all_e[r + 1]
            bi = 1.0 / max(temps[r], 1e-12)
            bj = 1.0 / max(temps[r + 1], 1e-12)
            log_accept = (bi - bj) * (ei - ej)
            if log_accept >= 0.0 or np.random.random() < math.exp(log_accept):
                for i in range(n):
                    tx = all_c[r, i, 0]
                    ty = all_c[r, i, 1]
                    all_c[r, i, 0] = all_c[r + 1, i, 0]
                    all_c[r, i, 1] = all_c[r + 1, i, 1]
                    all_c[r + 1, i, 0] = tx
                    all_c[r + 1, i, 1] = ty
                for a in range(gs):
                    for b in range(gs):
                        tg = all_g[r, a, b]
                        all_g[r, a, b] = all_g[r + 1, a, b]
                        all_g[r + 1, a, b] = tg
                te = all_e[r]
                all_e[r] = all_e[r + 1]
                all_e[r + 1] = te
            r += 2

    best_e = energy_full(best_c, seq, n)
    return best_e, best_c


In [7]:
def _sa_worker(args):
    return run_sa_numba(*args)


def _remc_worker(args):
    return run_remc_numba(*args)


def _safe_pool_map(func, args, workers):
    if workers <= 1 or len(args) <= 1:
        return [func(a) for a in args]
    try:
        method = "fork" if platform.system() != "Windows" else "spawn"
        ctx = get_context(method)
        with ctx.Pool(processes=workers) as pool:
            return pool.map(func, args)
    except Exception as exc:
        print(f"[pool] multiprocessing failed, falling back to serial: {exc}")
        traceback.print_exc(limit=1)
        return [func(a) for a in args]


def checkpoint_path(sid):
    return CHECKPOINT_DIR / f"best_{sid}.json"


def make_result(sid, coords, elapsed_seconds=0.0):
    spec = SEQUENCES[sid]
    seq_str = spec["sequence"]
    ok, info = validate_coords(seq_str, coords, verbose=False)
    if not ok:
        return {
            "sequence_id": sid,
            "valid": False,
            "error": info["error"],
            "coords": None,
        }
    contacts = int(info["contacts"])
    target = int(spec["best"])
    if contacts > target:
        status = "NEW_RECORD"
    elif contacts == target:
        status = "MATCHED_BEST_KNOWN"
    else:
        status = f"GAP_{target - contacts}"
    return {
        "sequence_id": sid,
        "lattice": "2D",
        "sequence": seq_str,
        "length": len(seq_str),
        "target_contacts": target,
        "contacts": contacts,
        "energy": -contacts,
        "radius_gyration2": float(info["radius_gyration2"]),
        "status": status,
        "elapsed_seconds": float(elapsed_seconds),
        "coords": [list(map(int, xy)) for xy in coords],
    }


def save_checkpoint(result):
    if not result or not result.get("valid", True):
        return
    path = checkpoint_path(result["sequence_id"])
    with path.open("w") as f:
        json.dump(result, f, indent=2)
    print(
        f"[checkpoint] {path} contacts={result['contacts']} "
        f"status={result['status']}"
    )


def load_checkpoint(sid):
    path = checkpoint_path(sid)
    if not path.exists():
        return None
    try:
        data = json.loads(path.read_text())
        seq_str = SEQUENCES[sid]["sequence"]
        ok, info = validate_coords(seq_str, data.get("coords"), verbose=False)
        if not ok:
            print(f"[checkpoint] ignoring invalid {path}: {info['error']}")
            return None
        data["contacts"] = int(info["contacts"])
        data["energy"] = -int(info["contacts"])
        data["radius_gyration2"] = float(info["radius_gyration2"])
        print(f"[checkpoint] loaded {sid}: contacts={data['contacts']}")
        return data
    except Exception as exc:
        print(f"[checkpoint] could not load {path}: {exc}")
        return None


def write_submission(result):
    sid = result["sequence_id"]
    if result.get("coords") is None:
        raise ValueError(f"{sid}: no coordinates to submit")
    seq_str = SEQUENCES[sid]["sequence"]
    ok, info = validate_coords(seq_str, result["coords"], verbose=False)
    if not ok:
        raise ValueError(f"{sid}: invalid fold, not writing submission: {info}")
    result["contacts"] = int(info["contacts"])
    result["energy"] = -int(info["contacts"])
    result["radius_gyration2"] = float(info["radius_gyration2"])
    sub = {
        "sequence_id": sid,
        "lattice": "2D",
        "coords": result["coords"],
    }
    path = RESULT_DIR / f"{sid}_submission.json"
    with path.open("w") as f:
        json.dump(sub, f, indent=2)
    print(f"[submission] wrote {path}")


def plot_fold(result):
    if not HAS_MPL or result.get("coords") is None:
        return
    sid = result["sequence_id"]
    seq_str = result["sequence"]
    coords = np.asarray(result["coords"], dtype=np.int32)
    fig, ax = plt.subplots(figsize=(8.5, 8.5))
    ax.set_aspect("equal")
    ax.plot(coords[:, 0], coords[:, 1], "-", color="#b8b8b8", lw=1.5, zorder=1)
    for i in range(len(seq_str)):
        if seq_str[i] != "H":
            continue
        for j in range(i + 2, len(seq_str)):
            if seq_str[j] != "H":
                continue
            if abs(coords[j, 0] - coords[i, 0]) + abs(coords[j, 1] - coords[i, 1]) == 1:
                ax.plot(
                    [coords[i, 0], coords[j, 0]],
                    [coords[i, 1], coords[j, 1]],
                    "--",
                    color="#d94841",
                    lw=1.7,
                    alpha=0.55,
                    zorder=2,
                )
    for i, (x, y) in enumerate(coords):
        color = "#1f9d72" if seq_str[i] == "H" else "#3478c7"
        ax.add_patch(Circle((x, y), 0.35, fc=color, ec="white", lw=1.2, zorder=3))
        ax.text(x, y, str(i), ha="center", va="center", fontsize=4.5, color="white", zorder=4)
    ax.set_title(
        f"{sid}: contacts={result['contacts']} target={result['target_contacts']} "
        f"{result['status']}",
        fontsize=10,
    )
    pad = 2
    ax.set_xlim(coords[:, 0].min() - pad, coords[:, 0].max() + pad)
    ax.set_ylim(coords[:, 1].min() - pad, coords[:, 1].max() + pad)
    ax.grid(True, alpha=0.13)
    path = FIGURE_DIR / f"{sid}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


In [8]:
@dataclass
class SolverConfig:
    total_hours: float = 9.0
    sequence_ids: tuple = tuple(SUBMIT_IDS)
    max_workers: int = max(1, min(cpu_count(), 16))
    strict_official_check: bool = True
    resume: bool = True
    smoke: bool = False
    random_seed: int = 20260529


TIME_WEIGHTS = {
    "S5": 0.8,
    "S6": 0.8,
    "S7": 1.1,
    "S8": 1.3,
    "S9": 2.8,
    "S10": 3.0,
}


def _sequence_hours(config: SolverConfig):
    ids = list(config.sequence_ids)
    total_weight = sum(TIME_WEIGHTS.get(sid, 1.0) for sid in ids)
    return {
        sid: config.total_hours * TIME_WEIGHTS.get(sid, 1.0) / total_weight
        for sid in ids
    }


def solve_sequence(sid: str, hours: float, workers: int, resume: bool = True, smoke: bool = False):
    if sid not in SEQUENCES:
        raise KeyError(f"Unknown sequence id: {sid}")
    spec = SEQUENCES[sid]
    seq_str = spec["sequence"]
    target = int(spec["best"])
    n = len(seq_str)
    seq = encode_sequence(seq_str)

    print("\n" + "=" * 78)
    print(f"[solve] {sid} length={n} target={target} budget={hours:.3f}h workers={workers}")
    print("=" * 78)

    best_result = load_checkpoint(sid) if resume else None
    if best_result is not None:
        best_coords = np.asarray(best_result["coords"], dtype=np.int32)
        best_e = -int(best_result["contacts"])
    else:
        best_coords = np.zeros((n, 2), dtype=np.int32)
        best_e = 0

    deadline = time.time() + max(1.0, hours * 3600.0)
    start_time = time.time()
    round_no = 0
    workers = max(1, int(workers))

    while time.time() < deadline:
        remaining = deadline - time.time()
        if smoke and round_no >= 1:
            break
        if remaining < (5.0 if smoke else 35.0):
            break
        round_no += 1

        if smoke:
            sa_chains = max(2, min(workers, 4))
            sa_steps = 1500
            remc_runs = max(1, min(workers, 2))
            nrep = 4
            spe = 80
            nex = 4
        else:
            sa_chains = min(max(8, workers * 8), 256)
            if n >= 85:
                sa_chains = min(max(6, workers * 6), 192)
            sa_steps = min(4_000_000, max(200_000, int(remaining * 1200)))
            remc_runs = min(max(2, workers), 12)
            nrep = min(40, max(12, n // 3))
            spe = min(8000, max(1000, n * 55))
            nex = min(1200, max(100, int(remaining * 0.25)))

        print(
            f"[round {round_no}] SA {sa_chains}x{sa_steps} "
            f"then REMC {remc_runs}x{nrep}rep x {nex}ex x {spe}steps "
            f"remaining={remaining/60:.1f}min"
        )

        seeds = np.random.randint(1, 2**31 - 1, size=sa_chains)
        empty_start = np.zeros((n, 2), dtype=np.int32)
        sa_args = []
        for i in range(sa_chains):
            use_seed = best_e < 0 and i == 0
            start_c = best_coords if use_seed else empty_start
            sa_args.append(
                (seq, n, sa_steps, 5.5, 0.002, int(seeds[i]), start_c, bool(use_seed))
            )
        sa_results = _safe_pool_map(_sa_worker, sa_args, min(workers, sa_chains))
        improved = False
        for e, c in sa_results:
            if e < best_e and valid_walk(c, n):
                best_e = int(e)
                best_coords = np.asarray(c, dtype=np.int32).copy()
                improved = True
        if improved:
            best_result = make_result(sid, best_coords, time.time() - start_time)
            print(f"[round {round_no}] SA improved: contacts={-best_e}")
            save_checkpoint(best_result)
        else:
            print(f"[round {round_no}] SA best so far: contacts={-best_e}")

        if time.time() >= deadline:
            break

        seeds = np.random.randint(1, 2**31 - 1, size=remc_runs)
        remc_args = []
        empty_start = np.zeros((n, 2), dtype=np.int32)
        for i in range(remc_runs):
            use_seed = best_e < 0 and i == 0
            start_c = best_coords if use_seed else empty_start
            remc_args.append(
                (
                    seq,
                    n,
                    nrep,
                    spe,
                    nex,
                    int(seeds[i]),
                    0.035,
                    6.0,
                    start_c,
                    bool(use_seed),
                )
            )
        remc_results = _safe_pool_map(_remc_worker, remc_args, min(workers, remc_runs))
        improved = False
        for e, c in remc_results:
            if e < best_e and valid_walk(c, n):
                best_e = int(e)
                best_coords = np.asarray(c, dtype=np.int32).copy()
                improved = True
        if improved:
            best_result = make_result(sid, best_coords, time.time() - start_time)
            print(f"[round {round_no}] REMC improved: contacts={-best_e}")
            save_checkpoint(best_result)
        else:
            print(f"[round {round_no}] REMC best so far: contacts={-best_e}")

        if -best_e > target:
            print(f"[round {round_no}] new record candidate found; continuing to improve")
        elif -best_e == target:
            print(f"[round {round_no}] matched current best-known; continuing to seek record")

    if best_e >= 0:
        # Guaranteed-valid fallback, so every output path remains testable.
        c, _, _, _, _ = init_snake(n)
        best_coords = np.asarray(c, dtype=np.int32)
        best_e = int(energy_full(best_coords, seq, n))

    best_result = make_result(sid, best_coords, time.time() - start_time)
    ok, info = validate_coords(seq_str, best_result["coords"], verbose=True)
    if not ok:
        raise RuntimeError(f"{sid}: invalid final fold: {info}")
    save_checkpoint(best_result)
    write_submission(best_result)
    plot_fold(best_result)
    print(
        f"[done] {sid}: contacts={best_result['contacts']} "
        f"target={target} status={best_result['status']}"
    )
    return best_result


def run_competition(config: SolverConfig):
    random.seed(config.random_seed)
    np.random.seed(config.random_seed)
    if config.strict_official_check:
        check_official_problem_data(strict=True)
    else:
        check_official_problem_data(strict=False)

    # JIT warmup and sanity tests before spending the real budget.
    run_self_tests()

    hours_by_sid = _sequence_hours(config)
    results = []
    for sid in config.sequence_ids:
        result = solve_sequence(
            sid,
            hours=hours_by_sid[sid],
            workers=config.max_workers,
            resume=config.resume,
            smoke=config.smoke,
        )
        results.append(result)

    write_verification_artifact(results, config)
    print_summary(results)
    return results


def write_verification_artifact(results, config: SolverConfig):
    artifact = {
        "conference": "CAISc 2026",
        "track": "Verifiable Problems",
        "problem": "HP Protein Folding",
        "problem_url": OFFICIAL_PROBLEM_URL,
        "method": "CPU Numba simulated annealing + replica exchange + pivot/crankshaft/rebridging moves",
        "gpu_used": False,
        "cpu_workers": int(config.max_workers),
        "total_hours_requested": float(config.total_hours),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "python": sys.version,
        "platform": platform.platform(),
        "results": results,
    }
    path = RESULT_DIR / "verification.json"
    with path.open("w") as f:
        json.dump(artifact, f, indent=2)
    print(f"[artifact] wrote {path}")


def print_summary(results):
    print("\n" + "=" * 78)
    print("FINAL SUMMARY")
    print("=" * 78)
    print(f"{'ID':<5} {'Len':>4} {'Target':>7} {'Ours':>6} {'Rg2':>10}  Status")
    print("-" * 78)
    for r in results:
        print(
            f"{r['sequence_id']:<5} {r['length']:>4} {r['target_contacts']:>7} "
            f"{r['contacts']:>6} {r['radius_gyration2']:>10.4f}  {r['status']}"
        )
    print("=" * 78)


In [9]:
def run_self_tests():
    print("[self-test] starting")
    for sid, spec in SEQUENCES.items():
        seq = spec["sequence"]
        assert set(seq) <= {"H", "P"}, sid
        assert len(seq) == len(spec["sequence"]), sid

    warm = "HHHHHH"
    warm_coords = [[0, 0], [1, 0], [1, 1], [0, 1], [0, 2], [1, 2]]
    ok, info = validate_coords(warm, warm_coords, expected_contacts=2, verbose=False)
    assert ok and info["contacts"] == 2

    seq = encode_sequence(SEQUENCES["S1"]["sequence"])
    n = len(seq)
    empty = np.zeros((n, 2), dtype=np.int32)
    e, c = run_sa_numba(seq, n, 300, 2.0, 0.1, 12345, empty, False)
    assert valid_walk(c, n)
    assert e == energy_full(c, seq, n)
    e2, c2 = run_remc_numba(seq, n, 4, 40, 3, 23456, 0.1, 3.0, c, True)
    assert valid_walk(c2, n)
    assert e2 == energy_full(c2, seq, n)
    print("[self-test] OK")


## Full Competition Run

Recommended Lightning AI setting:
- Use a CPU-rich instance. This notebook does not use the A100 GPU.
- Keep `MAX_WORKERS` at or below the physical CPU count.
- For a one-account run, start with 8-10 total hours and let checkpointing
  preserve every improvement.

To run a quick health check only, set `SMOKE = True`.


In [10]:
def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--hours", type=float, default=9.0)
    parser.add_argument("--seq", nargs="*", default=SUBMIT_IDS)
    parser.add_argument("--workers", type=int, default=max(1, min(cpu_count(), 16)))
    parser.add_argument("--no-strict-official-check", action="store_true")
    parser.add_argument("--no-resume", action="store_true")
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args(argv)

    config = SolverConfig(
        total_hours=args.hours,
        sequence_ids=tuple(args.seq),
        max_workers=args.workers,
        strict_official_check=not args.no_strict_official_check,
        resume=not args.no_resume,
        smoke=args.smoke,
    )
    return run_competition(config)


## Quick Health Check

Run this cell before the full run. It compiles the Numba kernels and verifies the scorer.


In [11]:
check_official_problem_data(strict=True)
run_self_tests()


[official-check] OK: local sequences/targets match live CAISc JSON
[self-test] starting


[self-test] OK


## Launch Competition Run

This is the long run. It uses CPU workers, writes checkpoints after improvements, and validates each JSON before writing it.


In [ ]:
# Final run settings for Lightning AI.
# GPU is intentionally not used; this run uses CPU workers on the instance.
RUN_FULL_COMPETITION = True
TOTAL_HOURS = 10.0
SEQUENCE_IDS = tuple(SUBMIT_IDS)
MAX_WORKERS = max(1, min(cpu_count(), 16))
RESUME = True
SMOKE = False

if RUN_FULL_COMPETITION:
    config = SolverConfig(
        total_hours=TOTAL_HOURS,
        sequence_ids=SEQUENCE_IDS,
        max_workers=MAX_WORKERS,
        strict_official_check=True,
        resume=RESUME,
        smoke=SMOKE,
    )
    results = run_competition(config)
else:
    print("Set RUN_FULL_COMPETITION = True when ready to launch the long run.")


[official-check] OK: local sequences/targets match live CAISc JSON
[self-test] starting
[self-test] OK

[solve] S5 length=48 target=23 budget=0.816h workers=16
[round 1] SA 128x3526530 then REMC 12x16rep x 734ex x 2640steps remaining=49.0min


[round 1] SA improved: contacts=22
[checkpoint] checkpoints/best_S5.json contacts=22 status=GAP_1
[round 1] REMC improved: contacts=23
[checkpoint] checkpoints/best_S5.json contacts=23 status=MATCHED_BEST_KNOWN
[round 1] matched current best-known; continuing to seek record
[round 2] SA 128x3486994 then REMC 12x16rep x 726ex x 2640steps remaining=48.4min
[round 2] SA best so far: contacts=23
[round 2] REMC best so far: contacts=23
[round 2] matched current best-known; continuing to seek record
[round 3] SA 128x3448040 then REMC 12x16rep x 718ex x 2640steps remaining=47.9min
[round 3] SA best so far: contacts=23
[round 3] REMC best so far: contacts=23
[round 3] matched current best-known; continuing to seek record
[round 4] SA 128x3409498 then REMC 12x16rep x 710ex x 2640steps remaining=47.4min
[round 4] SA best so far: contacts=23
[round 4] REMC best so far: contacts=23
[round 4] matched current best-known; continuing to seek record
[round 5] SA 128x3371204 then REMC 12x16rep x 702ex x